In [ ]:
# =====================================
# STEP 1: Install Kaggle
# =====================================
!pip install -q kaggle

# =====================================
# STEP 2: Upload kaggle.json
# =====================================
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# =====================================
# STEP 3: Download Dataset
# =====================================
!kaggle datasets download -d jessicali9530/lfw-dataset
!unzip -q lfw-dataset.zip

# =====================================
# STEP 4: Imports
# =====================================
import os
import cv2
import numpy as np
import tensorflow as tf
from datetime import datetime
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

print("Imports OK \u2705")

# =====================================
# STEP 5: Create folders for tracking
# =====================================
os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# =====================================
# STEP 6: Load Data (MORE DATA -> HIGH ACCURACY)
# =====================================
DATA_DIR = "lfw-deepfunneled/lfw-deepfunneled"

faces = []
labels = []
label_map = {}
label_count = 0

for person in os.listdir(DATA_DIR):
    person_path = os.path.join(DATA_DIR, person)

    # take only meaningful classes
    if len(os.listdir(person_path)) < 20:
        continue

    label_map[label_count] = person

    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.resize(img, (224, 224))
        faces.append(img)
        labels.append(label_count)

    label_count += 1

faces = np.array(faces, dtype="float32") / 255.0
labels = np.array(labels)

print("Dataset:", faces.shape)
print("Classes:", len(label_map))

# =====================================
# STEP 7: Manual Split
# =====================================
indices = np.arange(len(faces))
np.random.shuffle(indices)

faces = faces[indices]
labels = labels[indices]

split = int(0.8 * len(faces))

X_train = faces[:split]
X_test = faces[split:]
y_train = labels[:split]
y_test = labels[split:]

# =====================================
# STEP 8: High Accuracy Model
# =====================================
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# freeze most layers but allow top fine-tuning
for layer in base_model.layers[:-20]:
    layer.trainable = False

for layer in base_model.layers[-20:]:
    layer.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(len(label_map), activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.00005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# =====================================
# STEP 9: Data Augmentation
# =====================================
datagen = ImageDataGenerator(
    rotation_range=25,
    zoom_range=0.3,
    horizontal_flip=True
)

# =====================================
# STEP 10: Train (MORE EPOCHS)
# =====================================
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=15,
    validation_data=(X_test, y_test)
)

# =====================================
# STEP 11: Evaluate
# =====================================
loss, acc = model.evaluate(X_test, y_test)
accuracy_percent = acc * 100
print("Final Accuracy:", accuracy_percent)

# =====================================
# STEP 12: Save Model with VERSION
# =====================================
existing_models = len(os.listdir("models")) + 1
model_name = f"models/face_model_v{existing_models}_acc_{accuracy_percent:.2f}.h5"

model.save(model_name)
print("Saved:", model_name)

# =====================================
# STEP 13: Save Training Log
# =====================================
log_file = f"logs/log_v{existing_models}.txt"

with open(log_file, "w") as f:
    f.write(f"Model Version: {existing_models}\n")
    f.write(f"Accuracy: {accuracy_percent}\n")
    f.write(f"Epochs: 15\n")
    f.write(f"Classes: {len(label_map)}\n")
    f.write(f"Time: {datetime.now()}\n")

print("Log saved:", log_file)

# =====================================
# STEP 14: Plot
# =====================================
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train','Validation'])
plt.title("Accuracy")
plt.show()
